<a href="https://colab.research.google.com/github/lilydeller/Math344-Project/blob/main/notebooks/04_what_it_learned.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 1

Clone repository and import functions.

In [5]:
import sys

USER = "lilydeller"
REPO = "Math344-Project"

!rm -rf {REPO}
!git clone -q https://github.com/{USER}/{REPO}.git

sys.path.insert(0, f"{REPO}/src")

from evaluate import to_matrix
from models import baseline_matrix, rank_k

Download and import the movie dataset with movie titles.

In [6]:
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

ML_DATASET = Path("ml-100k")

if not ML_DATASET.exists():
    url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
    response = requests.get(url)

    ZipFile(BytesIO(response.content)).extractall()

movie_ratings = pd.read_csv(
    ML_DATASET / "u.data", sep="\t", names=["user", "item", "rating", "time"]
)

titles = pd.read_csv(
    ML_DATASET / "u.item", sep="|", encoding="latin-1", header=None, usecols=[1]
)[1].to_numpy()

print(len(titles), titles[49])

1682 Star Wars (1977)


# Part 2

Create ratings matrix from full dataset and using it to calculate if a movie is popular it has at least 50 ratings.

In [7]:
R = to_matrix(movie_ratings)
known = ~np.isnan(R)
B = baseline_matrix(R)
X = np.where(known, R - B, 0.0)
U, s, Vt = np.linalg.svd(X, full_matrices=False)

n_ratings = known.sum(axis=0)
popular = n_ratings >= 50
print("popular movies: ", popular.sum())

popular movies:  603


# Part 3

Sorts a row of movies' scores, calculated based on a hidden factor, in ascending order. Then, it prints the first/top and last/bottom n scores alongside each movie title, representing the two contrasting ends identified by the factor.

In [10]:
def show_factor(f, n=6):
#Popular movies at the two ends of hidden factor f (f = 1, 2, ...)
  v = Vt[f - 1]
  idx = np.where(popular)[0]
  order = idx[np.argsort(v[idx])]
  print(f"Factor {f}, one end:")
  for m in order[:n]:
    print(f" {v[m]:+.3f} {titles[m]}")
  print(f"Factor {f}, other end:")
  for m in order[::-1][:n]:
    print(f" {v[m]:+.3f} {titles[m]}")

show_factor(2, 8)

Factor 2, one end:
 -0.099 Independence Day (ID4) (1996)
 -0.092 Twister (1996)
 -0.077 Eraser (1996)
 -0.074 Air Force One (1997)
 -0.067 Saint, The (1997)
 -0.066 Phenomenon (1996)
 -0.062 Ghost and the Darkness, The (1996)
 -0.061 Murder at 1600 (1997)
Factor 2, other end:
 +0.203 Star Wars (1977)
 +0.199 Pulp Fiction (1994)
 +0.173 Godfather, The (1972)
 +0.168 Empire Strikes Back, The (1980)
 +0.151 Fargo (1996)
 +0.146 Return of the Jedi (1983)
 +0.134 2001: A Space Odyssey (1968)
 +0.133 Jaws (1975)
